# Realizations from an angular power spectrum

This notebook generates random sky maps from a simple angular power spectrum

$$
\frac{\ell^2 C_\ell}{2\pi} = 1 + A\exp\left[-\frac{1}{2}\left(\frac{\ell-\ell_c}{\sigma}\right)^2\right].
$$

The left panel shows the input spectrum. The right panel shows one random realization of that spectrum on the sphere. Re-running the plotting cell with a different random seed gives a different sky with the same statistical power spectrum.

## 1. Setup

This cell is written to work in Google Colab. Colab usually has NumPy, SciPy, and Matplotlib already installed, but it may need `healpy` for spherical harmonic map synthesis.

In [ ]:
try:
    import healpy as hp
except ImportError:
    %pip -q install healpy
    import healpy as hp

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from healpy.projaxes import HpxMollweideAxes
from urllib.request import urlopen

try:
    import ipywidgets as widgets
except ImportError:
    %pip -q install ipywidgets
    import ipywidgets as widgets

from IPython.display import clear_output, display

mpl.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 13,
    "axes.linewidth": 0.8,
})

## 2. Load the Planck-style colormap

We use Matthew Petroff's CLASS CMB colormap, which is designed to resemble the Planck CMB map colormap while improving perceptual behavior. The colormap file registers a Matplotlib colormap named `"class"`.

In [ ]:
COLORMAP_URL = "https://cdn0.mpetroff.net/wp-content/uploads/2023/05/colormap.py"

if "class" not in mpl.colormaps:
    colormap_source = urlopen(COLORMAP_URL).read().decode("utf-8")
    exec(colormap_source, {})

cmb_cmap = mpl.colormaps["class"]

## 3. Define the spectrum and map-making function

The function below takes the bump amplitude `A`, center `ell_c`, and width `sigma`. It builds the corresponding $C_\ell$, draws a Gaussian random realization with `healpy.synfast`, and plots both the spectrum and the map.

A few implementation notes:

- The plotted spectrum is $D_\ell = \ell^2 C_\ell/(2\pi)$.
- We set the monopole and dipole, $\ell=0,1$, to zero because the expression is intended for angular fluctuations.
- `nside` controls the map resolution; `lmax` controls the largest multipole included.

In [ ]:
def make_power_spectrum(A, ell_c, sigma, lmax=256):
    """Return ell, C_ell, and D_ell for the toy power spectrum.

    Parameters
    ----------
    A : float
        Amplitude of the Gaussian bump in D_ell.
    ell_c : float
        Multipole where the Gaussian bump is centered.
    sigma : float
        Width of the Gaussian bump in multipole space.
    lmax : int
        Maximum multipole to include.
    """
    ell = np.arange(lmax + 1, dtype=float)
    D_ell = np.zeros_like(ell)

    use = ell >= 2
    D_ell[use] = 1.0 + A * np.exp(-0.5 * ((ell[use] - ell_c) / sigma) ** 2)

    C_ell = np.zeros_like(ell)
    C_ell[use] = 2.0 * np.pi * D_ell[use] / ell[use] ** 2
    return ell, C_ell, D_ell


def plot_spectrum_and_realization(A=4.0, ell_c=80.0, sigma=12.0, *, lmax=256, nside=128, seed=1):
    """Plot the toy power spectrum and one random sky realization.

    Change A, ell_c, sigma, or seed to explore how the spectrum affects the map.
    """
    ell, C_ell, D_ell = make_power_spectrum(A, ell_c, sigma, lmax=lmax)

    # healpy.synfast uses NumPy's global random state in many versions,
    # so setting the seed here makes the example reproducible in Colab.
    np.random.seed(seed)
    sky = hp.synfast(C_ell, nside=nside, lmax=lmax, new=True)

    # In Colab, replacing the old output helps keep the page from jumping
    # when this cell is evaluated repeatedly with different parameters.
    clear_output(wait=True)

    vmax = np.percentile(np.abs(sky), 99.5)
    fig = plt.figure(figsize=(12, 4.8))

    # Use explicit axes positions instead of tight_layout(), since healpy
    # creates its own projection axes and can trigger layout warnings.
    ax = fig.add_axes([0.07, 0.16, 0.38, 0.68])
    ax.plot(ell[2:], D_ell[2:], color="black", lw=2)
    ax.axvline(ell_c, color="0.5", ls="--", lw=1)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$\ell^2 C_\ell/(2\pi)$")
    ax.set_title("Input power spectrum")
    ax.grid(alpha=0.3)
    ax.set_xlim(2, lmax)
    ax.set_ylim(0, 1.15 * np.max(D_ell))

    map_ax = HpxMollweideAxes(fig, [0.54, 0.16, 0.40, 0.68])
    fig.add_axes(map_ax)
    map_ax.projmap(sky, nest=False, vmin=-vmax, vmax=vmax, cmap=cmb_cmap)
    map_ax.set_title("One realization")

    fig.suptitle(rf"$A={A:g}$, $\ell_c={ell_c:g}$, $\sigma={sigma:g}$, seed={seed}", y=0.97)
    plt.show()

    return ell, C_ell, D_ell, sky

## 4. Try one example

The map is random, so changing `seed` changes the particular realization. The broad features should still reflect the same input power spectrum.

In [ ]:
ell, C_ell, D_ell, sky = plot_spectrum_and_realization(
    A=4.0,
    ell_c=80.0,
    sigma=12.0,
    lmax=256,
    nside=128,
    seed=3,
)

## 5. A few things to explore

- Increase `A`: the preferred angular scale becomes more visually prominent.
- Increase `ell_c`: the map features move to smaller angular scales.
- Increase `sigma`: the bump becomes broader in multipole space, mixing a wider range of angular scales.
- Change `seed`: the exact sky changes, but the statistical pattern is drawn from the same spectrum.

## 6. Try a pure power-law spectrum

This cell uses

$$
C_\ell = \left(\frac{\ell}{50}\right)^n.
$$

This fixes the amplitude to be $C_{50}=1$ for every value of the slope. The slope `n` controls whether power is weighted toward large angular scales, small angular scales, or distributed more evenly. In Google Colab, the `#@param` comments create editable controls in the cell.

In [ ]:
def make_power_law_spectrum(n, lmax=250, pivot_ell=50.0):
    """Return ell and C_ell for C_ell = (ell / pivot_ell)^n.

    With the default pivot_ell=50, every spectrum has C_50 = 1.
    The monopole and dipole are set to zero so the map shows fluctuations
    rather than an arbitrary mean or large dipole pattern.
    """
    ell = np.arange(lmax + 1, dtype=float)
    C_ell = np.zeros_like(ell)
    use = ell >= 2
    C_ell[use] = (ell[use] / pivot_ell) ** n
    return ell, C_ell


def plot_power_law_realization(n=-2.0, *, lmax=250, nside=128, seed=7):
    """Plot C_ell = (ell/50)^n and one random realization."""
    ell, C_ell = make_power_law_spectrum(n, lmax=lmax)

    np.random.seed(seed)
    sky = hp.synfast(C_ell, nside=nside, lmax=lmax, new=True)

    clear_output(wait=True)

    vmax = np.percentile(np.abs(sky), 99.5)
    fig = plt.figure(figsize=(12, 4.8))

    ax = fig.add_axes([0.07, 0.16, 0.38, 0.68])
    ax.loglog(ell[2:], C_ell[2:], color="black", lw=2)
    ax.axvline(50, color="0.5", ls="--", lw=1)
    ax.axhline(1, color="0.5", ls=":", lw=1)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$C_\ell$")
    ax.set_title(r"Power law: $C_\ell=(\ell/50)^n$")
    ax.grid(alpha=0.3, which="both")
    ax.set_xlim(2, lmax)
    ax.set_ylim(1e-3, 5e5)

    map_ax = HpxMollweideAxes(fig, [0.54, 0.16, 0.40, 0.68])
    fig.add_axes(map_ax)
    map_ax.projmap(sky, nest=False, vmin=-vmax, vmax=vmax, cmap=cmb_cmap)
    map_ax.set_title("One realization")

    fig.suptitle(rf"$C_\ell=(\ell/50)^{{{n:g}}}$, seed={seed}", y=0.97)
    plt.show()

    return ell, C_ell, sky

## 7. Interactive slope slider

Use this slider to see how the power-law slope changes both the spectrum and a corresponding realization.

In [ ]:
def show_power_law_with_slope(n=-2.0):
    """Update the power-law spectrum and one matching realization."""
    plot_power_law_realization(n=n, lmax=250, nside=128, seed=7)


n_slider = widgets.FloatSlider(
    value=-2.0,
    min=-4.0,
    max=2.0,
    step=0.1,
    description="n",
    continuous_update=False,
)

interactive_spectrum = widgets.interactive_output(
    show_power_law_with_slope,
    {"n": n_slider},
)

display(n_slider)
display(interactive_spectrum)